In [7]:
import pandas as pd
import numpy as np

In [17]:
df = pd.read_parquet('data/cleaned_reviews.parquet')

In [19]:
df[
    df["clean_authors"]
    .apply(lambda x: any("-" in a.lower() for a in x))
]

,Id,Title,Price,User_id,profileName,score,time,summary,text,description,...,previewLink,publisher,publishedDate,infoLink,categories,ratingsCount,clean_text,clean_summary,clean_categories,clean_authors


In [4]:
def eh_desconhecido(x):
    if x is None: return True
    if isinstance(x, (list, np.ndarray)):
        return len(x) == 0 or 'unknown' in x
    return str(x).lower() in ["unknown", "['unknown']", "[]"]

# 3. Cria o gabarito pegando apenas títulos que possuem autores REAIS
gabarito = (
    df[~df["authors"].apply(eh_desconhecido)][["Title", "authors"]]
    .drop_duplicates(subset=["Title"])
    .rename(columns={"authors": "authors_gabarito"})
)

# 4. Faz o cruzamento para testar a recuperação
df_recuperado = df.merge(gabarito, on="Title", how="left")

# 5. Filtra quem era desconhecido mas achou um par no gabarito
linhas_salvas = df_recuperado[
    df_recuperado["authors"].apply(eh_desconhecido) & 
    df_recuperado["authors_gabarito"].notna()
]

print(f"Total de registros que conseguimos recuperar: {len(linhas_salvas)}")
if len(linhas_salvas) > 0:
    print("\n--- Exemplos de Títulos Recuperados (Antes vs Depois) ---")
    print(linhas_salvas[["Title", "authors", "authors_gabarito"]].drop_duplicates(subset=["Title"]).head(10))
else:
    print("\nPoxa, nenhum título cruzou. Significa que os livros com 'unknown' são exclusivos e não possuem duplicatas com autor preenchido.")

Total de registros que conseguimos recuperar: 0

Poxa, nenhum título cruzou. Significa que os livros com 'unknown' são exclusivos e não possuem duplicatas com autor preenchido.


In [5]:
df['clean_authors'].explode().value_counts()

clean_authors
unknown                 390842
jane austen              37511
j r r tolkien            37302
charles dickens          21878
emily bronte             18445
                         ...  
eric william allison         1
nolan dennett                1
dr brian fagan               1
chris scarre                 1
bob bowersox                 1
Name: count, Length: 151297, dtype: int64

In [6]:
# 1. Filtra as linhas onde 'unknown' está presente na lista
df_filtrado = df[df['clean_authors'].apply(lambda x: 'unknown' in x if isinstance(x, (list, np.ndarray)) else False)][['authors', 'clean_authors']]

# 2. Converte para tupla apenas o que for iterável, preservando nulos, e roda o distinct
df_filtrado.map(lambda x: tuple(x) if isinstance(x, (list, np.ndarray)) else x).drop_duplicates()

,authors,clean_authors
74,None,"(unknown,)"


In [7]:
df['categories'].explode().value_counts()

categories
Fiction                            824439
unknown                            551498
Juvenile Fiction                   207542
Biography & Autobiography          107791
Religion                            98035
                                    ...  
Action and adventure films              1
Mormon temples                          1
Alternative veterinary medicine         1
Arkansas River                          1
Statesmen's spouses                     1
Name: count, Length: 10884, dtype: int64

In [8]:
df_categorie_normalize = pd.read_parquet('data/cleaned_reviews_mapped.parquet')

In [9]:
df_categorie_normalize['categories_clean'].value_counts()

categories_clean
fiction                       845901
juvenile fiction              213825
history                       126627
religion                      112783
biography  autobiography      110010
business  economics            75807
computers                      49194
family  relationships          40905
cooking                        34815
unknown                        33258
education                      32669
social science                 32636
young adult fiction            31858
self-help                      29636
juvenile nonfiction            29484
body mind  spirit              29359
health  fitness                28517
psychology                     28000
science                        27867
travel                         25618
philosophy                     25336
sports  recreation             25006
political science              24318
art                            24262
pets                           22645
adventure stories              20803
drama                

In [10]:
df_pares = df_categorie_normalize[['categories', 'categories_clean']].drop_duplicates()
amostra_notebook = df_pares.sample(n=20, random_state=42)

# Exibe a amostra para você avaliar visualmente
for i, row in enumerate(amostra_notebook.itertuples(), 1):
    print(f"{i}. Original: '{row.categories}'  ==>  Predito: '{row.categories_clean}'")

1. Original: 'electronic government information'  ==>  Predito: 'computers'
2. Original: 'invasion from mars radio play'  ==>  Predito: 'music'
3. Original: 'soccer'  ==>  Predito: 'sports  recreation'
4. Original: 'funeral homes'  ==>  Predito: 'unknown'
5. Original: 'macram'  ==>  Predito: 'unknown'
6. Original: 'interfaith marriage'  ==>  Predito: 'family  relationships'
7. Original: 'south dakota'  ==>  Predito: 'education'
8. Original: 'atlanta ga'  ==>  Predito: 'unknown'
9. Original: 'shock'  ==>  Predito: 'humor'
10. Original: 'doll clothes'  ==>  Predito: 'crafts  hobbies'
11. Original: 'eighteenth century'  ==>  Predito: 'juvenile fiction'
12. Original: 'contests'  ==>  Predito: 'games  activities'
13. Original: 'australians'  ==>  Predito: 'pets'
14. Original: 'amazon river region'  ==>  Predito: 'unknown'
15. Original: 'psychiatric hospital care'  ==>  Predito: 'medical'
16. Original: 'book of mormon'  ==>  Predito: 'bibles'
17. Original: 'local author'  ==>  Predito: 'amer

1. Original: 'electronic government information'  ==>  Predito: 'computers' ✅
2. Original: 'invasion from mars radio play'  ==>  Predito: 'music' ❌
3. Original: 'soccer'  ==>  Predito: 'sports  recreation' ✅
4. Original: 'funeral homes'  ==>  Predito: 'unknown'❌
5. Original: 'macram'  ==>  Predito: 'unknown'❌
6. Original: 'interfaith marriage'  ==>  Predito: 'family  relationships' ✅
7. Original: 'south dakota'  ==>  Predito: 'education'❌
8. Original: 'atlanta ga'  ==>  Predito: 'unknown'❌
9. Original: 'shock'  ==>  Predito: 'humor'❌
10. Original: 'doll clothes'  ==>  Predito: 'crafts  hobbies' ✅
11. Original: 'eighteenth century'  ==>  Predito: 'juvenile fiction'❌
12. Original: 'contests'  ==>  Predito: 'games  activities'❌
13. Original: 'australians'  ==>  Predito: 'pets'❌
14. Original: 'amazon river region'  ==>  Predito: 'unknown'❌
15. Original: 'psychiatric hospital care'  ==>  Predito: 'medical' ✅
16. Original: 'book of mormon'  ==>  Predito: 'bibles' ✅
17. Original: 'local author'  ==>  Predito: 'american literature'❌
18. Original: 'saltcellars'  ==>  Predito: 'unknown'❌
19. Original: 'investments'  ==>  Predito: 'business  economics' ✅
20. Original: 'buddhist logic'  ==>  Predito: 'philosophy' ✅


Assertividade duvidosa, necessario melhor analise de threshold ou avaliação de mtodo para similaridade(talvez outra LLM)

In [24]:
len(df_categorie_normalize)

2448502

In [23]:
len(df_categorie_normalize[df_categorie_normalize['User_id'] != 'Unknown'])

1985586

In [20]:
df_data_sentiment = pd.read_parquet('data/data_sentiment1.parquet')
df_data_sentiment[df_data_sentiment['ratingsCount'] >= 30].head(100)

,Id,Title,Price,User_id,profileName,score,time,summary,text,description,...,categories_clean,sentiment_score,review_length,unique_words,aspect_plot,aspect_style,aspect_pacing,sentiment_text,text_embedding,clean_authors
14,0595344550,Whispers of the Wicked Saints,10.95,A3Q12RK71N74LB,Book Reader,1.0,1117065600,not good,I bought this book because I read some glowing...,Julia Thomas finds her life spinning out of co...,...,fiction,negativo,105,88,0.339467,0.094270,0.331437,positivo,"[-0.0761291, -0.020619342, -0.029056199, 0.009...",[veronica haddon]
15,0595344550,Whispers of the Wicked Saints,10.95,A1E9M6APK30ZAU,V. Powell,4.0,1119571200,Here is my opinion,"I have to admit, I am not one to write reviews...",Julia Thomas finds her life spinning out of co...,...,fiction,positivo,101,77,0.296438,0.208742,0.333124,neutro,"[-0.07262105, -0.033631608, -0.006486672, 0.06...",[veronica haddon]
16,0595344550,Whispers of the Wicked Saints,10.95,AUR0VA5H0C66C,"LoveToRead ""Actually Read Books""",1.0,1119225600,Buyer beware,"This is a self-published book, and if you want...",Julia Thomas finds her life spinning out of co...,...,fiction,negativo,72,62,0.230171,0.081367,0.178687,neutro,"[-0.063394226, -0.021854093, 0.037728418, 0.11...",[veronica haddon]
17,0595344550,Whispers of the Wicked Saints,10.95,A1YLDZ3VHR6QPZ,Clara,5.0,1115942400,Fall on your knee's,When I first read this the I was mezmerized at...,Julia Thomas finds her life spinning out of co...,...,fiction,positivo,30,30,0.361481,0.243496,0.395484,neutro,"[-0.014930362, 0.011321619, 0.0016866978, 0.07...",[veronica haddon]
18,0595344550,Whispers of the Wicked Saints,10.95,ACO23CG8K8T77,Tonya,5.0,1117065600,Bravo Veronica,I read the review directly under mine and I ha...,Julia Thomas finds her life spinning out of co...,...,fiction,positivo,59,46,0.193113,0.182724,0.311852,neutro,"[-0.061996296, -0.0017075611, -0.09379933, 0.0...",[veronica haddon]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428,0671551345,Night World: Daughters Of Darkness,Unknown,AKLGZKOXH9B9E,K1mca,5.0,947203200,Daughters of darkness the best of night world,i think that this is probebly to me the best n...,"""There’s something strange about the new girls...",...,juvenile fiction,positivo,38,26,0.392932,0.063749,0.239084,positivo,"[-0.024958096, 0.013438815, 0.008018052, 0.043...",[lj smith]
429,0671551345,Night World: Daughters Of Darkness,Unknown,Unknown,Unknown,5.0,961459200,One of the best books I've ever read!,DAUGHTERS OF DARKNESS was the first book by L....,"""There’s something strange about the new girls...",...,juvenile fiction,positivo,0,0,0.000000,0.000000,0.000000,neutro,None,[lj smith]
430,0671551345,Night World: Daughters Of Darkness,Unknown,A16O2SAOUUH2QM,Rebecca,5.0,939340800,The BEST book of the Night World series!!!!!,This book was great and I just loved Ash. It s...,"""There’s something strange about the new girls...",...,juvenile fiction,positivo,25,21,0.405097,0.066682,0.255537,positivo,"[-0.073076054, 0.021466695, 0.023145638, 0.038...",[lj smith]
431,0671551345,Night World: Daughters Of Darkness,Unknown,A38EDDASAM1W0V,M. Cookson,5.0,1022976000,best of the Night World books,"Jade, Kestrel, and Rowan, three vampire sister...","""There’s something strange about the new girls...",...,juvenile fiction,positivo,137,105,0.231076,0.033980,0.036027,positivo,"[-0.010860526, -0.05009736, 0.00096952164, 0.0...",[lj smith]
